# Notebook 2: Model Training

---

**Author**: Laura Steel Pascual  
**Date**: February 2026  
**Project**: Decoding Anthropogenic Impact with eDNA and Supervised Machine Learning: Species‑Level Associations from Solomon Island Rivers

---

## Overview
This notebook trains and evaluates supervised learning classifiers to predict site environmental status (reference vs impacted) from eDNA species composition.

**Purpose**: 
- Train 4 classifier types across 8 buffer-threshold combinations.
- Compare classifier performance.
- Optimise hyperparameters for best-performing classifier.
- Save trained models for model validation and feature importance analyses.

**Methods:**
- Classifiers: Decision Tree, Random Forest, XGBoost, Logistic Regression.
- Pipeline: log(1+x) transformation -> Z-score normalisation -> classification.
- Cross-validation: Stratified 5-fold (see Notebook 2a for 10-fold comparison).
- Hyperparameter tuning: GridSearchCV on Random Forest (n_estimators, max_depth).
- Metrics: AUC, Accuracy, Precision, Recall, F1.

**Paper Reference**: 2. Materials & Methods -> 2.4 Machine Learning Classification, 2.5 Model Selection and Hyperparameter Tuning; 3. Results -> 3.1 Classifier Performance and Selection, 3.2 Optimised Random Forest Performance and Statistical Validation

---

## 1. Setup

In [3]:
# Import libraries
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import pickle

# Create output directories
os.makedirs('results/models', exist_ok=True)
os.makedirs('results/model_performance', exist_ok=True)

print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Scikit-learn version: {sklearn.__version__}')
print(f'Xgboost version: {xgboost.__version__}')
print(f'Matplotlib version: {matplotlib.__version__}')
print(f'Seaborn version: {sns.__version__}')

NumPy version: 1.24.4
Pandas version: 2.3.2
Scikit-learn version: 1.2.2
Xgboost version: 3.0.3
Matplotlib version: 3.9.2
Seaborn version: 0.12.2


## 2. Load Preprocessed Datasets

Load the 8 preprocessed pickle files (4 buffers × 2 thresholds) from Notebook 1 containing X_raw, y, feature_names, and metadata.

**Note**: Raw eDNA counts loaded. Transformations occur within ML pipelines. For rationale see Notebook 1.

In [5]:
# Configuration: 8 datasets (4 buffers × 2 thresholds)
buffers = [100, 300, 500, 1000]
thresholds = [0.03125, 0.0625]

datasets = {}
for buf in buffers:
    for thresh in thresholds:
        thresh_str = str(thresh).replace('.', '')
        filename = f'data_{buf}m_t{thresh_str}.pkl'
        filepath = os.path.join('results/preprocessed', filename)
        
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        
        key = f'{buf}m_t{thresh_str}'
        datasets[key] = data
        
        print(f'{key}: {data["n_reference"]} reference, {data["n_impacted"]} impacted')

print(f'\nLoaded {len(datasets)} datasets')
print(f'Features per dataset: {len(datasets["100m_t003125"]["feature_names"])}')

100m_t003125: 26 reference, 21 impacted
100m_t00625: 27 reference, 20 impacted
300m_t003125: 22 reference, 25 impacted
300m_t00625: 25 reference, 22 impacted
500m_t003125: 16 reference, 31 impacted
500m_t00625: 23 reference, 24 impacted
1000m_t003125: 18 reference, 29 impacted
1000m_t00625: 29 reference, 18 impacted

Loaded 8 datasets
Features per dataset: 861


## 3. Define ML Pipeline and Classifiers

Create consistent pipeline across all classifiers to ensure fair comparison.
- Classifiers: Decision Tree, Random Forest, XGBoost, Logistic Regression.
- CV Strategy: 5-fold CV (see Notebook 2a for 10-fold comparison).

In [7]:
# Define log1p transformer
log_transformer = FunctionTransformer(np.log1p, validate=True)

# 5-fold stratified cross-validation
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define classifiers
classifiers = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(random_state=42, n_jobs=-1)
}

print(f'Classifiers: {list(classifiers.keys())}')
print(f'CV strategy: 5-fold stratified')

Classifiers: ['Decision Tree', 'Random Forest', 'XGBoost', 'Logistic Regression']
CV strategy: 5-fold stratified


## 4. Classifier Benchmarking: 5-Fold CV with Default Parameters

Initial performance assessment using 5-fold stratified cross-validation with default hyperparameters. Due to limited sample size (n=47), 5-fold CV provides more stable estimates than 10-fold (see Notebook 2a for comparison).

In [9]:
results_5fold = {}

for dataset_key, data in datasets.items():
    print(f'\n{"="*80}')
    print(f'Dataset: {dataset_key}')
    print(f'{"="*80}')
    
    X_raw = data['X_raw']  # numpy array
    y = data['y']  # numpy array
    feature_names = data['feature_names']
    
    results_5fold[dataset_key] = {}
    
    for clf_name, clf in classifiers.items():
        print(f"\nClassifier: {clf_name}")
        
        pipe = Pipeline([
            ('log_transform', log_transformer),
            ('scaler', StandardScaler()),
            ('clf', clf)
        ])

        preds = np.zeros_like(y, dtype=int)
        accuracy_scores = []
        auc_scores = []
        feature_importances = []
        train_accuracy_scores = []
        train_auc_scores = []

        for fold_idx, (train_idx, test_idx) in enumerate(cv_strategy.split(X_raw, y)):
            X_train, X_test = X_raw[train_idx], X_raw[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            pipe.fit(X_train, y_train)

            y_pred = pipe.predict(X_test)
            if hasattr(pipe.named_steps['clf'], "predict_proba"):
                y_proba = pipe.predict_proba(X_test)[:, 1]
                y_train_proba = pipe.predict_proba(X_train)[:, 1]
            else:
                y_proba = pipe.decision_function(X_test)
                y_train_proba = pipe.decision_function(X_train)

            y_train_pred = pipe.predict(X_train)
            preds[test_idx] = y_pred

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            try:
                auc_scores.append(roc_auc_score(y_test, y_proba))
                train_auc_scores.append(roc_auc_score(y_train, y_train_proba))
            except Exception:
                auc_scores.append(np.nan)
                train_auc_scores.append(np.nan)
            
            train_accuracy_scores.append(accuracy_score(y_train, y_train_pred))

            # Feature importances (if available)
            if hasattr(pipe.named_steps['clf'], 'feature_importances_'):
                importances = pipe.named_steps['clf'].feature_importances_
                feature_importances.append(importances)

        # Print summary statistics
        print(f"Mean Train Accuracy: {np.mean(train_accuracy_scores):.4f}, Std: {np.std(train_accuracy_scores):.4f}")
        print(f"Mean Train AUC-ROC: {np.nanmean(train_auc_scores):.4f}, Std: {np.nanstd(train_auc_scores):.4f}")
        print(f"Mean Validation Accuracy: {np.mean(accuracy_scores):.4f}, Std: {np.std(accuracy_scores):.4f}")
        print(f"Mean Validation AUC-ROC: {np.nanmean(auc_scores):.4f}, Std: {np.nanstd(auc_scores):.4f}")
        
        # Classification report
        print(f"Classification report:")
        print(classification_report(y, preds))

        # Store results
        results_5fold[dataset_key][clf_name] = {
            'train_accuracy': train_accuracy_scores,
            'test_accuracy': accuracy_scores,
            'train_auc': train_auc_scores,
            'test_auc': auc_scores,
            'predictions': preds,
            'feature_importances': feature_importances
        }

print("\n5-fold CV benchmarking complete")


Dataset: 100m_t003125

Classifier: Decision Tree
Mean Train Accuracy: 1.0000, Std: 0.0000
Mean Train AUC-ROC: 1.0000, Std: 0.0000
Mean Validation Accuracy: 0.6133, Std: 0.1408
Mean Validation AUC-ROC: 0.6100, Std: 0.1632
Classification report:
              precision    recall  f1-score   support

           0       0.64      0.69      0.67        26
           1       0.58      0.52      0.55        21

    accuracy                           0.62        47
   macro avg       0.61      0.61      0.61        47
weighted avg       0.61      0.62      0.61        47


Classifier: Random Forest
Mean Train Accuracy: 1.0000, Std: 0.0000
Mean Train AUC-ROC: 1.0000, Std: 0.0000
Mean Validation Accuracy: 0.5289, Std: 0.1181
Mean Validation AUC-ROC: 0.6513, Std: 0.1883
Classification report:
              precision    recall  f1-score   support

           0       0.56      0.77      0.65        26
           1       0.45      0.24      0.31        21

    accuracy                           0.5

### 4.1 Create Summary Tables and Export Results

Compile results into summary table focusing on AUC and Accuracy.

In [11]:
summary_rows = []

for dataset_key in datasets.keys():
    # Parse dataset key to extract buffer and threshold
    parts = dataset_key.split('_')
    buffer = int(parts[0].replace('m', ''))
    threshold_str = parts[1].replace('t', '')
    threshold = float('0.' + threshold_str)
    
    for clf_name in classifiers.keys():
        result = results_5fold[dataset_key][clf_name]
        
        summary_rows.append({
            'Buffer (m)': buffer,
            'Threshold': threshold,
            'Dataset': dataset_key,
            'Classifier': clf_name,
            'Train Accuracy': np.mean(result['train_accuracy']),
            'Train Accuracy Std': np.std(result['train_accuracy']),
            'Test Accuracy': np.mean(result['test_accuracy']),
            'Test Accuracy Std': np.std(result['test_accuracy']),
            'Train AUC': np.nanmean(result['train_auc']),
            'Train AUC Std': np.nanstd(result['train_auc']),
            'Test AUC': np.nanmean(result['test_auc']),
            'Test AUC Std': np.nanstd(result['test_auc'])
        })

summary_df = pd.DataFrame(summary_rows)

# Display test performance only
print("\n5-Fold CV Performance Summary (Test Set):")
print("="*90)
test_cols = ['Buffer (m)', 'Threshold', 'Classifier', 'Test Accuracy', 'Test Accuracy Std', 
             'Test AUC', 'Test AUC Std']
print(summary_df[test_cols].to_string(index=False))

# Save full table to Excel (includes both train and test)
summary_df.to_excel('results/model_performance/5fold_cv_summary.xlsx', index=False)
print("\n" + "="*90)
print("Saved: results/model_performance/5fold_cv_summary.xlsx")


5-Fold CV Performance Summary (Test Set):
 Buffer (m)  Threshold          Classifier  Test Accuracy  Test Accuracy Std  Test AUC  Test AUC Std
        100   0.003125       Decision Tree       0.613333           0.140791  0.610000      0.163248
        100   0.003125       Random Forest       0.528889           0.118134  0.651333      0.188345
        100   0.003125             XGBoost       0.640000           0.148341  0.682333      0.182342
        100   0.003125 Logistic Regression       0.591111           0.135974  0.636667      0.161383
        100   0.006250       Decision Tree       0.548889           0.165343  0.551667      0.181460
        100   0.006250       Random Forest       0.633333           0.134440  0.630833      0.177161
        100   0.006250             XGBoost       0.746667           0.192886  0.770000      0.209669
        100   0.006250 Logistic Regression       0.655556           0.128812  0.670000      0.121861
        300   0.003125       Decision Tree      

### 4.2 Export Class-Wise Metrics

Extract precision, recall, and F1-score separately for Class 0 (Reference) and Class 1 (Impacted) sites, along with support counts for each class. This allows detailed assessment of model performance on each class independently.

In [13]:
classwise_rows = []

for dataset_key in datasets.keys():
    # Parse dataset key
    parts = dataset_key.split('_')
    buffer = int(parts[0].replace('m', ''))
    threshold_str = parts[1].replace('t', '')
    threshold = float('0.' + threshold_str)
    
    for clf_name in classifiers.keys():
        result = results_5fold[dataset_key][clf_name]
        
        # Get predictions
        y = datasets[dataset_key]['y']
        preds = result['predictions']
        
        # Calculate class-wise metrics
        from sklearn.metrics import precision_recall_fscore_support
        precision, recall, f1, support = precision_recall_fscore_support(
            y, preds, average=None, zero_division=0
        )
        
        # Add rows for each class
        for class_idx in [0, 1]:
            class_label = 'Reference' if class_idx == 0 else 'Impacted'
            
            classwise_rows.append({
                'Buffer (m)': buffer,
                'Threshold': threshold,
                'Dataset': dataset_key,
                'Classifier': clf_name,
                'Class': class_idx,
                'Class_Label': class_label,
                'Precision': precision[class_idx],
                'Recall': recall[class_idx],
                'F1-Score': f1[class_idx],
                'Support': support[class_idx],
                'Mean_Test_Accuracy': np.mean(result['test_accuracy']),
                'Mean_Test_AUC': np.nanmean(result['test_auc'])
            })

classwise_df = pd.DataFrame(classwise_rows)

# Display format
print("\n" + "="*100)
print("Class-wise Performance Metrics (5-Fold CV):")
print("="*100)

for dataset_key in datasets.keys():
    print(f"\n{dataset_key}:")
    print("-" * 90)
    
    dataset_data = classwise_df[classwise_df['Dataset'] == dataset_key]
    
    for clf_name in classifiers.keys():
        clf_data = dataset_data[dataset_data['Classifier'] == clf_name]
        if len(clf_data) > 0:
            print(f"\n  {clf_name}:")
            print(f"    {'Class':<6} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support':<10}")
            print(f"    {'-'*52}")
            for _, row in clf_data.iterrows():
                print(f"    {row['Class']:<6} {row['Precision']:<12.2f} "
                      f"{row['Recall']:<12.2f} {row['F1-Score']:<12.2f} {row['Support']:<10.0f}")
            
            # Print mean metrics
            mean_acc = clf_data.iloc[0]['Mean_Test_Accuracy']
            mean_auc = clf_data.iloc[0]['Mean_Test_AUC']
            print(f"    Mean Test Accuracy: {mean_acc:.4f}")
            print(f"    Mean Test AUC: {mean_auc:.4f}")

# Save to Excel
classwise_df.to_excel('results/model_performance/5fold_cv_classwise_metrics.xlsx', index=False)
print("\n\nSaved: results/model_performance/5fold_cv_classwise_metrics.xlsx")


Class-wise Performance Metrics (5-Fold CV):

100m_t003125:
------------------------------------------------------------------------------------------

  Decision Tree:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.64         0.69         0.67         26        
    1      0.58         0.52         0.55         21        
    Mean Test Accuracy: 0.6133
    Mean Test AUC: 0.6100

  Random Forest:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.56         0.77         0.65         26        
    1      0.45         0.24         0.31         21        
    Mean Test Accuracy: 0.5289
    Mean Test AUC: 0.6513

  XGBoost:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.66         0.73         0.69         26        
    1      0.61         0.52

### 4.3 Variance Analysis

Quantify CV stability to justify the choice of 5-fold over 10-fold CV (see Notebook 2a for implementation).

In [15]:
# Extract variance metrics for comparison
variance_comparison = []

for dataset_key in datasets.keys():
    for clf_name in classifiers.keys():
        result = results_5fold[dataset_key][clf_name]
        
        variance_comparison.append({
            'Dataset': dataset_key,
            'Classifier': clf_name,
            'CV_Folds': 5,
            'Acc_Mean': np.mean(result['test_accuracy']),
            'Acc_Std': np.std(result['test_accuracy']),
            'AUC_Mean': np.nanmean(result['test_auc']),
            'AUC_Std': np.nanstd(result['test_auc'])
        })

variance_df = pd.DataFrame(variance_comparison)

# Summary statistics
print("\n" + "="*90)
print("Variance Analysis: 5-Fold CV Stability")
print("="*90)
print(f"\nMean AUC Std across all configurations: {variance_df['AUC_Std'].mean():.4f}")
print(f"Mean Accuracy Std across all configurations: {variance_df['Acc_Std'].mean():.4f}")
print(f"\nConfigurations with AUC Std > 0.20: {(variance_df['AUC_Std'] > 0.20).sum()}/{len(variance_df)}")
print(f"Configurations with Accuracy Std > 0.20: {(variance_df['Acc_Std'] > 0.20).sum()}/{len(variance_df)}")

# Save to Excel
variance_df.to_excel('results/model_performance/5fold_variance_analysis.xlsx', index=False)
print("\nSaved: results/model_performance/5fold_variance_analysis.xlsx")


Variance Analysis: 5-Fold CV Stability

Mean AUC Std across all configurations: 0.1380
Mean Accuracy Std across all configurations: 0.1209

Configurations with AUC Std > 0.20: 3/32
Configurations with Accuracy Std > 0.20: 0/32

Saved: results/model_performance/5fold_variance_analysis.xlsx


### 4.4 Best Classifier Per Combination

Identify which classifier achieved the best performance for each buffer-threshold combination using a combined score (average of Test AUC and Test Accuracy). This helps determine if one classifier consistently outperforms others across configurations, although it is not the only determinant.

In [17]:
print("\n" + "="*100)
print("Best Classifier per Buffer-Threshold Combination:")
print("="*100)

# Create combined score (average of normalized AUC and Accuracy)
summary_df['Combined Score'] = (summary_df['Test AUC'] + summary_df['Test Accuracy']) / 2

for (buffer, threshold), group in summary_df.groupby(['Buffer (m)', 'Threshold']):
    best_row = group.loc[group['Combined Score'].idxmax()]
    
    # Get class-wise metrics for the winner
    winner_classwise = classwise_df[
        (classwise_df['Buffer (m)'] == buffer) & 
        (classwise_df['Threshold'] == threshold) & 
        (classwise_df['Classifier'] == best_row['Classifier'])
    ]
    
    class_0 = winner_classwise[winner_classwise['Class'] == 0].iloc[0]
    class_1 = winner_classwise[winner_classwise['Class'] == 1].iloc[0]
    
    print(f"\n{buffer}m, Threshold={threshold}:")
    print(f"  Winner: {best_row['Classifier']}")
    print(f"  Test AUC: {best_row['Test AUC']:.4f} ± {best_row['Test AUC Std']:.4f}")
    print(f"  Test Accuracy: {best_row['Test Accuracy']:.4f} ± {best_row['Test Accuracy Std']:.4f}")
    print(f"  Class 0 (Reference) - Precision: {class_0['Precision']:.2f}, "
          f"Recall: {class_0['Recall']:.2f}, F1: {class_0['F1-Score']:.2f}")
    print(f"  Class 1 (Impacted)  - Precision: {class_1['Precision']:.2f}, "
          f"Recall: {class_1['Recall']:.2f}, F1: {class_1['F1-Score']:.2f}")
    print(f"  Combined Score: {best_row['Combined Score']:.4f}")

# Count wins per classifier
print("\n" + "="*100)
print("Classifier win count (by Combined Score: AUC + Accuracy):")
print("="*100)
wins = summary_df.loc[summary_df.groupby(['Buffer (m)', 'Threshold'])['Combined Score'].idxmax()]['Classifier'].value_counts()
print(wins)
overall_winner = wins.idxmax()
print(f"\nOverall winner: {overall_winner} ({wins[overall_winner]}/8 combinations)")


Best Classifier per Buffer-Threshold Combination:

100m, Threshold=0.003125:
  Winner: XGBoost
  Test AUC: 0.6823 ± 0.1823
  Test Accuracy: 0.6400 ± 0.1483
  Class 0 (Reference) - Precision: 0.66, Recall: 0.73, F1: 0.69
  Class 1 (Impacted)  - Precision: 0.61, Recall: 0.52, F1: 0.56
  Combined Score: 0.6612

100m, Threshold=0.00625:
  Winner: XGBoost
  Test AUC: 0.7700 ± 0.2097
  Test Accuracy: 0.7467 ± 0.1929
  Class 0 (Reference) - Precision: 0.76, Recall: 0.81, F1: 0.79
  Class 1 (Impacted)  - Precision: 0.72, Recall: 0.65, F1: 0.68
  Combined Score: 0.7583

300m, Threshold=0.003125:
  Winner: Random Forest
  Test AUC: 0.7740 ± 0.1274
  Test Accuracy: 0.7044 ± 0.1204
  Class 0 (Reference) - Precision: 0.72, Recall: 0.59, F1: 0.65
  Class 1 (Impacted)  - Precision: 0.69, Recall: 0.80, F1: 0.74
  Combined Score: 0.7392

300m, Threshold=0.00625:
  Winner: Random Forest
  Test AUC: 0.7970 ± 0.1419
  Test Accuracy: 0.7000 ± 0.1778
  Class 0 (Reference) - Precision: 0.68, Recall: 0.84, F

## 5. Hyperparameter Tuning: Random Forest with GridSearchCV

Random Forest shows strongest performance overall. Optimise hyperparameters using GridSearchCV with 5-fold CV.
- Parameter ranges: n_estimators [100, 200, 300, 500, 700, 1000], max_depth [2, 3, 4, 5, 7, None].
- class_weight='balanced' to address class imbalance.
- All other parameters default.

Refit criterion: AUC (primary metric).

In [19]:
def tune_rf_with_metrics(dataset_key, data):
    """
    Tune Random Forest hyperparameters using GridSearchCV.
    
    Parameters:
    -----------
    dataset_key : str
        Dataset identifier (e.g., '100m_t003125')
    data : dict
        Preprocessed data dictionary from pickle file
    
    Returns:
    --------
    Pipeline : Best estimator from GridSearchCV
    """
    print(f"\nTuning Random Forest for {dataset_key}")
    
    X = data['X_raw']  # numpy array
    y = data['y']  # numpy array

    pipe = Pipeline([
        ('log_transform', log_transformer),
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            max_features='sqrt',  # default
            bootstrap=True,  # default
            class_weight='balanced',  # addressing imbalance
            min_samples_split=2,  # default
            min_samples_leaf=1  # default
        ))
    ])

    param_grid = {
        'clf__n_estimators': [100, 200, 300, 500, 700, 1000],
        'clf__max_depth': [2, 3, 4, 5, 7, None],
    }

    scoring = {
        'AUC': 'roc_auc',
        'Accuracy': 'accuracy'
    }

    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=5,
        scoring=scoring,
        refit='AUC',  # refit best model by AUC
        n_jobs=-1,
        verbose=2,
        return_train_score=True
    )

    grid_search.fit(X, y)

    # Extract relevant average metrics
    print("Best parameters:", grid_search.best_params_)
    print(f"Best mean validation AUC: {grid_search.best_score_:.4f}")
    cv_results = grid_search.cv_results_

    mean_train_acc = np.max(cv_results['mean_train_Accuracy'])
    mean_valid_acc = np.max(cv_results['mean_test_Accuracy'])
    mean_train_auc = np.max(cv_results['mean_train_AUC'])
    mean_valid_auc = np.max(cv_results['mean_test_AUC'])

    print(f"Mean Train Accuracy (best params): {mean_train_acc:.4f}")
    print(f"Mean Validation Accuracy (best params): {mean_valid_acc:.4f}")
    print(f"Mean Train AUC (best params): {mean_train_auc:.4f}")
    print(f"Mean Validation AUC (best params): {mean_valid_auc:.4f}")

    return grid_search.best_estimator_

print('Hyperparameter tuning function defined')

Hyperparameter tuning function defined


## 6. Tune Random Forest for All Buffer/Threshold Combinations

Best Configurations (from GridSearchCV output):
- 100m / 0.03125: max_depth=7, n_estimators=500
- 100m / 0.0625: max_depth=5, n_estimators=300
- 300m / 0.03125: max_depth=4, n_estimators=200
- 300m / 0.0625: max_depth=4, n_estimators=700
- 500m / 0.03125: max_depth=3, n_estimators=100
- 500m / 0.0625: max_depth=3, n_estimators=200
- 1000m / 0.03125: max_depth=2, n_estimators=500
- 1000m / 0.0625: max_depth=4, n_estimators=200

In [21]:
# Tune for all 8 configurations (4 buffers × 2 thresholds)
best_models = {}

for dataset_key, data in datasets.items():
    best_models[dataset_key] = tune_rf_with_metrics(dataset_key, data)

print("\nHyperparameter tuning complete for all configurations")


Tuning Random Forest for 100m_t003125
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters: {'clf__max_depth': 7, 'clf__n_estimators': 500}
Best mean validation AUC: 0.6167
Mean Train Accuracy (best params): 1.0000
Mean Validation Accuracy (best params): 0.5578
Mean Train AUC (best params): 1.0000
Mean Validation AUC (best params): 0.6167

Tuning Random Forest for 100m_t00625
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters: {'clf__max_depth': 5, 'clf__n_estimators': 300}
Best mean validation AUC: 0.5767
Mean Train Accuracy (best params): 1.0000
Mean Validation Accuracy (best params): 0.5556
Mean Train AUC (best params): 1.0000
Mean Validation AUC (best params): 0.5767

Tuning Random Forest for 300m_t003125
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters: {'clf__max_depth': 4, 'clf__n_estimators': 200}
Best mean validation AUC: 0.6860
Mean Train Accuracy (best params): 1.0000
Mean Validation Accuracy (

### 6.1 Evaluate Optimised Random Forest Models with 5-Fold CV

Re-evaluate the tuned Random Forest models using 5-fold CV to get complete performance metrics (fold-level variance, class-wise metrics) for the optimised hyperparameters, and to ensure consistency with the benchmarking section. Additionally, OOB (out-of-bag) scores are collected, which provide an internal estimate of generalisation error specific to Random Forests via bootstrap sampling.

In [23]:
print("\n" + "="*90)
print("Evaluating Optimised Random Forest Models (5-Fold CV)")
print("="*90)

results_rf_optimised = {}

for dataset_key, data in datasets.items():
    print(f'\nDataset: {dataset_key}')
    
    X_raw = data['X_raw']  # numpy array
    y = data['y']  # numpy array
    feature_names = data['feature_names']
    
    # Get the optimised model for this dataset
    optimised_model = best_models[dataset_key]
    
    # Extract just the classifier parameters to create a fresh classifier
    clf_params = optimised_model.named_steps['clf'].get_params()
    
    # Enable OOB scoring
    clf_params['oob_score'] = True
    
    # Create fresh pipeline with optimised parameters
    pipe = Pipeline([
        ('log_transform', log_transformer),
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(**clf_params))
    ])
    
    preds = np.zeros_like(y, dtype=int)
    accuracy_scores = []
    auc_scores = []
    feature_importances = []
    train_accuracy_scores = []
    train_auc_scores = []
    oob_scores = []
    
    for fold_idx, (train_idx, test_idx) in enumerate(cv_strategy.split(X_raw, y)):
        X_train, X_test = X_raw[train_idx], X_raw[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        pipe.fit(X_train, y_train)
        
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
        y_train_pred = pipe.predict(X_train)
        
        preds[test_idx] = y_pred
        
        accuracy_scores.append(accuracy_score(y_test, y_pred))
        auc_scores.append(roc_auc_score(y_test, y_proba))
        train_auc_scores.append(roc_auc_score(y_train, y_train_proba))
        train_accuracy_scores.append(accuracy_score(y_train, y_train_pred))
        
        # Get OOB score
        oob_scores.append(pipe.named_steps['clf'].oob_score_)
        
        # Feature importances
        importances = pipe.named_steps['clf'].feature_importances_
        feature_importances.append(importances)
    
    # Print summary
    print(f"  Mean Test Accuracy: {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}")
    print(f"  Mean Test AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
    print(f"  Mean OOB Score: {np.mean(oob_scores):.4f} ± {np.std(oob_scores):.4f}")
    
    # Store results
    results_rf_optimised[dataset_key] = {
        'train_accuracy': train_accuracy_scores,
        'test_accuracy': accuracy_scores,
        'train_auc': train_auc_scores,
        'test_auc': auc_scores,
        'oob_scores': oob_scores,
        'predictions': preds,
        'feature_importances': feature_importances
    }

print("\nOptimised Random Forest evaluation complete")


Evaluating Optimised Random Forest Models (5-Fold CV)

Dataset: 100m_t003125
  Mean Test Accuracy: 0.5911 ± 0.1360
  Mean Test AUC: 0.7077 ± 0.1286
  Mean OOB Score: 0.6327 ± 0.0531

Dataset: 100m_t00625
  Mean Test Accuracy: 0.6756 ± 0.1608
  Mean Test AUC: 0.7433 ± 0.1351
  Mean OOB Score: 0.6055 ± 0.0727

Dataset: 300m_t003125
  Mean Test Accuracy: 0.6844 ± 0.1276
  Mean Test AUC: 0.7900 ± 0.1114
  Mean OOB Score: 0.5960 ± 0.0977

Dataset: 300m_t00625
  Mean Test Accuracy: 0.7422 ± 0.1410
  Mean Test AUC: 0.7720 ± 0.1405
  Mean OOB Score: 0.6061 ± 0.0639

Dataset: 500m_t003125
  Mean Test Accuracy: 0.6200 ± 0.1312
  Mean Test AUC: 0.6571 ± 0.2170
  Mean OOB Score: 0.6812 ± 0.0837

Dataset: 500m_t00625
  Mean Test Accuracy: 0.7467 ± 0.1019
  Mean Test AUC: 0.7860 ± 0.1332
  Mean OOB Score: 0.5639 ± 0.0388

Dataset: 1000m_t003125
  Mean Test Accuracy: 0.7867 ± 0.0952
  Mean Test AUC: 0.8856 ± 0.0830
  Mean OOB Score: 0.7765 ± 0.0273

Dataset: 1000m_t00625
  Mean Test Accuracy: 0.8044

### 6.2 Create Summary Table for optimised Random Forest

Compile optimised Random Forest results into summary table

In [25]:
# Compile optimised RF results into summary table
rf_summary_rows = []

for dataset_key in datasets.keys():
    # Parse dataset key
    parts = dataset_key.split('_')
    buffer = int(parts[0].replace('m', ''))
    threshold_str = parts[1].replace('t', '')
    threshold = float('0.' + threshold_str)
    
    result = results_rf_optimised[dataset_key]
    
    # Get hyperparameters
    clf_params = best_models[dataset_key].named_steps['clf'].get_params()
    
    rf_summary_rows.append({
        'Buffer (m)': buffer,
        'Threshold': threshold,
        'Dataset': dataset_key,
        'max_depth': clf_params['max_depth'],
        'n_estimators': clf_params['n_estimators'],
        'Train Accuracy': np.mean(result['train_accuracy']),
        'Train Accuracy Std': np.std(result['train_accuracy']),
        'Test Accuracy': np.mean(result['test_accuracy']),
        'Test Accuracy Std': np.std(result['test_accuracy']),
        'Train AUC': np.mean(result['train_auc']),
        'Train AUC Std': np.std(result['train_auc']),
        'Test AUC': np.mean(result['test_auc']),
        'Test AUC Std': np.std(result['test_auc']),
        'OOB Score': np.mean(result['oob_scores']),
        'OOB Score Std': np.std(result['oob_scores'])
    })

rf_summary_df = pd.DataFrame(rf_summary_rows)

# Display test performance
print("\nOptimised Random Forest Performance Summary (Test Set):")
print("="*120)
test_cols = ['Buffer (m)', 'Threshold', 'max_depth', 'n_estimators', 
             'Test Accuracy', 'Test Accuracy Std', 'Test AUC', 'Test AUC Std',
             'OOB Score', 'OOB Score Std']
print(rf_summary_df[test_cols].to_string(index=False))

# Save to Excel
rf_summary_df.to_excel('results/model_performance/rf_optimised_summary.xlsx', index=False)
print("\n" + "="*120)
print("Saved: results/model_performance/rf_optimised_summary.xlsx")


Optimised Random Forest Performance Summary (Test Set):
 Buffer (m)  Threshold  max_depth  n_estimators  Test Accuracy  Test Accuracy Std  Test AUC  Test AUC Std  OOB Score  OOB Score Std
        100   0.003125          7           500       0.591111           0.135974  0.707667      0.128589   0.632717       0.053062
        100   0.006250          5           300       0.675556           0.160770  0.743333      0.135134   0.605548       0.072742
        300   0.003125          4           200       0.684444           0.127580  0.790000      0.111355   0.596017       0.097694
        300   0.006250          4           700       0.742222           0.140967  0.772000      0.140485   0.606117       0.063949
        500   0.003125          3           100       0.620000           0.131168  0.657143      0.217025   0.681223       0.083726
        500   0.006250          3           200       0.746667           0.101932  0.786000      0.133207   0.563869       0.038751
       1000   0.003

### 6.3 Export Class-Wise Metrics for Optimised Random Forest

Extract class-wise metrics for optimised Random Forest models.

In [27]:
rf_classwise_rows = []

for dataset_key in datasets.keys():
    # Parse dataset key
    parts = dataset_key.split('_')
    buffer = int(parts[0].replace('m', ''))
    threshold_str = parts[1].replace('t', '')
    threshold = float('0.' + threshold_str)
    
    result = results_rf_optimised[dataset_key]
    
    # Get predictions
    y = datasets[dataset_key]['y']
    preds = result['predictions']
    
    # Calculate class-wise metrics
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, support = precision_recall_fscore_support(
        y, preds, average=None, zero_division=0
    )
    
    # Add rows for each class
    for class_idx in [0, 1]:
        class_label = 'Reference' if class_idx == 0 else 'Impacted'
        
        rf_classwise_rows.append({
            'Buffer (m)': buffer,
            'Threshold': threshold,
            'Dataset': dataset_key,
            'Class': class_idx,
            'Class_Label': class_label,
            'Precision': precision[class_idx],
            'Recall': recall[class_idx],
            'F1-Score': f1[class_idx],
            'Support': support[class_idx],
            'Mean_Test_Accuracy': np.mean(result['test_accuracy']),
            'Mean_Test_AUC': np.mean(result['test_auc'])
        })

rf_classwise_df = pd.DataFrame(rf_classwise_rows)

# Display format
print("\n" + "="*100)
print("Optimised Random Forest - Class-wise Performance Metrics:")
print("="*100)

for dataset_key in datasets.keys():
    print(f"\n{dataset_key}:")
    print("-" * 80)
    
    dataset_data = rf_classwise_df[rf_classwise_df['Dataset'] == dataset_key]
    
    print(f"  {'Class':<6} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support':<10}")
    print(f"  {'-'*52}")
    for _, row in dataset_data.iterrows():
        print(f"  {row['Class']:<6} {row['Precision']:<12.2f} "
              f"{row['Recall']:<12.2f} {row['F1-Score']:<12.2f} {row['Support']:<10.0f}")
    
    # Print mean metrics
    mean_acc = dataset_data.iloc[0]['Mean_Test_Accuracy']
    mean_auc = dataset_data.iloc[0]['Mean_Test_AUC']
    print(f"  Mean Test Accuracy: {mean_acc:.4f}")
    print(f"  Mean Test AUC: {mean_auc:.4f}")

# Save to Excel
rf_classwise_df.to_excel('results/model_performance/rf_optimised_classwise_metrics.xlsx', index=False)
print("\n\nSaved: results/model_performance/rf_optimised_classwise_metrics.xlsx")


Optimised Random Forest - Class-wise Performance Metrics:

100m_t003125:
--------------------------------------------------------------------------------
  Class  Precision    Recall       F1           Support   
  ----------------------------------------------------
  0      0.59         0.85         0.70         26        
  1      0.60         0.29         0.39         21        
  Mean Test Accuracy: 0.5911
  Mean Test AUC: 0.7077

100m_t00625:
--------------------------------------------------------------------------------
  Class  Precision    Recall       F1           Support   
  ----------------------------------------------------
  0      0.67         0.89         0.76         27        
  1      0.73         0.40         0.52         20        
  Mean Test Accuracy: 0.6756
  Mean Test AUC: 0.7433

300m_t003125:
--------------------------------------------------------------------------------
  Class  Precision    Recall       F1           Support   
  -----------------------

## 7. Save Optimised Models

Save the optimised Random Forest pipelines for use in: 
- Notebook 3: Permutation testing for statistical validation.
- Notebook 4: SHAP analysis and feature importance and interpretation.

In [29]:
for dataset_key, model in best_models.items():
    # Save the full pipeline (includes log transform, scaler, and RF classifier)
    model_filename = f'rf_optimised_{dataset_key}.pkl'
    model_filepath = os.path.join('results/models', model_filename)
    
    with open(model_filepath, 'wb') as f:
        pickle.dump(model, f)
    
    print(f"Saved: {model_filename}")

print("All 8 optimised Random Forest models saved to results/models/")

Saved: rf_optimised_100m_t003125.pkl
Saved: rf_optimised_100m_t00625.pkl
Saved: rf_optimised_300m_t003125.pkl
Saved: rf_optimised_300m_t00625.pkl
Saved: rf_optimised_500m_t003125.pkl
Saved: rf_optimised_500m_t00625.pkl
Saved: rf_optimised_1000m_t003125.pkl
Saved: rf_optimised_1000m_t00625.pkl
All 8 optimised Random Forest models saved to results/models/


## 8. Summary

**Outputs** (`results/model_performance/`, `results/models/`):
- **Benchmarking results**: `5fold_cv_summary.xlsx`, `5fold_cv_classwise_metrics.xlsx`, `5fold_variance_analysis.xlsx`
- **Optimised Random Forest results**: `rf_optimised_summary.xlsx`, `rf_optimised_classwise_metrics.xlsx`
- **Trained models** (8 pickle files, `results/models/`): `rf_optimised_100m_t003125.pkl` through `rf_optimised_1000m_t00625.pkl`

Each trained model pipeline contains:
- `log_transform`: FunctionTransformer(np.log1p)
- `scaler`: StandardScaler (fit on training folds only)
- `clf`: RandomForestClassifier with tuned `n_estimators` and `max_depth`, `class_weight='balanced'`

## Next Steps

- Notebook 3 (`3_Permutation_Tests.ipynb`): Statistical validation via permutation testing.
- Notebook 4 (`4_Feature_Importance.ipynb`): SHAP and Gini importance analysis for statistically significant models.
- Notebook 2a (`2a_CV_Fold_Comparison.ipynb`): Justification of 5-fold over 10-fold CV.
- Notebook 2b (`2b_Binary_vs_Counts_Comparison.ipynb`): Justification of count data over binary presence/absence.